# pandas Merge, Join, Concatenate & Compare 튜토리얼
> **참조**: [pandas 공식 문서 — Merging](https://pandas.pydata.org/docs/user_guide/merging.html)
> **데이터**: [classicmodels ERD 샘플](https://www.mysqltutorial.org/getting-started-with-mysql/mysql-sample-database/) CSV 파일 (Google Drive 마운트)

---

## 목차
| # | 메서드 | 핵심 사용 목적 |
|---|--------|--------------|
| 1 | `pd.concat()` | 행/열 방향으로 단순 이어붙이기 |
| 2 | `pd.merge()` — INNER | 양쪽에 모두 있는 행만 |
| 3 | `pd.merge()` — LEFT | 왼쪽 전부 + 매칭되는 오른쪽 |
| 4 | `pd.merge()` — RIGHT | 오른쪽 전부 + 매칭되는 왼쪽 |
| 5 | `pd.merge()` — OUTER | 양쪽 모두 보존 |
| 6 | 다중 키 & 체인 merge | 3개 이상 테이블 연결 |
| 7 | `DataFrame.join()` | 인덱스 기반 조인 |
| 8 | `pd.merge_ordered()` | 시계열 정렬 병합 |
| 9 | `pd.merge_asof()` | 근사값(asof) 병합 |
| 10 | `combine_first()` | 결측값 보완 |
| 11 | `compare()` | 두 DataFrame 차이 비교 |

---




---
## 0. 환경 설정 — Google Drive 마운트 & CSV 로드


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 35)

# Google Drive 안에 CSV 파일을 저장한 경로로 수정하세요
DATA = "/content/drive/MyDrive/Colab Notebooks/2026/이어드림스쿨6기/dataset/erd_sample"   # <-- 경로 수정

productlines = pd.read_csv(f"{DATA}/productlines.csv")
products     = pd.read_csv(f"{DATA}/products.csv")
offices      = pd.read_csv(f"{DATA}/offices.csv")
employees    = pd.read_csv(f"{DATA}/employees.csv")
customers    = pd.read_csv(f"{DATA}/customers.csv")
orders       = pd.read_csv(f"{DATA}/orders.csv",       parse_dates=["orderDate","shippedDate"])
orderdetails = pd.read_csv(f"{DATA}/orderdetails.csv")
payments     = pd.read_csv(f"{DATA}/payments.csv",     parse_dates=["paymentDate"])

print("CSV 로드 완료")
for name, df in [("productlines", productlines), ("products", products),
                 ("offices", offices), ("employees", employees),
                 ("customers", customers), ("orders", orders),
                 ("orderdetails", orderdetails), ("payments", payments)]:
    print(f"  {name:15s}: {df.shape[0]}행 x {df.shape[1]}열")


CSV 로드 완료
  productlines   : 7행 x 4열
  products       : 10행 x 9열
  offices        : 7행 x 9열
  employees      : 9행 x 8열
  customers      : 10행 x 13열
  orders         : 10행 x 7열
  orderdetails   : 10행 x 5열
  payments       : 10행 x 4열


In [3]:
# 각 테이블 미리보기
print("=== orders ===")
display(orders.head(3))
print("=== customers ===")
display(customers.head(3))
print("=== orderdetails ===")
display(orderdetails.head(3))


=== orders ===


,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,NaN,363
1,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,NaN,128
2,10102,2003-01-10,2003-01-18,2003-01-14,Shipped,NaN,181


=== customers ===


,customerNumber,customerName,contactLastName,contactFirstName,phone,addressLine1,addressLine2,city,state,postalCode,country,salesRepEmployeeNumber,creditLimit
0,103,Atelier graphique,Schmitt,Carine,40.32.2555,"54, rue Royale",NaN,Nantes,NaN,44000,France,1370,21000
1,112,Signal Gift Stores,King,Jean,7025551838,8489 Strong St.,NaN,Las Vegas,NV,83030,USA,1166,71800
2,114,"Australian Collectors, Co.",Ferguson,Peter,03 9520 4555,636 St Kilda Road,Level 3,Melbourne,Victoria,3004,Australia,1611,117300


=== orderdetails ===


,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber
0,10100,S18_1342,30,136.00,3
1,10100,S18_2238,50,101.26,2
2,10100,S24_1444,22,65.00,4


---
## 1. `pd.concat()` — 행/열 방향 이어붙이기

> **언제 쓰나?** 같은 구조의 데이터를 단순히 쌓거나 옆에 붙일 때
> (스키마가 동일한 분기별 데이터, 지역별 파일 등)

```python
pd.concat(objs, axis=0, join='outer', ignore_index=False, keys=None)
```

| 파라미터 | 기본값 | 설명 |
|----------|--------|------|
| `axis` | `0` | 0 = 행 방향(세로), 1 = 열 방향(가로) |
| `join` | `'outer'` | `'outer'`(합집합) / `'inner'`(교집합) |
| `ignore_index` | `False` | True면 인덱스를 0, 1, 2 ... 로 초기화 |
| `keys` | `None` | 출처를 나타내는 MultiIndex 레이블 |


In [8]:
# 코드
# axis = 0, 행 방향(세로로 쌓기)
# 시나리오 Q1/ Q2 주문 데이터를 하나로 합치기

orders_q1 = orders.loc[orders['orderDate'] < '2003-02-01', :].copy()
orders_q2 = orders.loc[orders['orderDate'] >= '2003-02-01', :].copy()

result = pd.concat([orders_q1, orders_q2], ignore_index=True)
result

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber
0,10100,2003-01-06,2003-01-13,2003-01-10,Shipped,NaN,363
1,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,NaN,128
2,10102,2003-01-10,2003-01-18,2003-01-14,Shipped,NaN,181
3,10103,2003-01-29,2003-02-07,2003-02-02,Shipped,NaN,121
4,10104,2003-01-31,2003-02-14,2003-01-29,Shipped,NaN,141
5,10105,2003-02-11,2003-02-21,2003-02-12,Shipped,NaN,145
6,10106,2003-02-17,2003-02-24,2003-02-21,Shipped,NaN,278
7,10107,2003-02-24,2003-03-03,2003-02-26,Shipped,NaN,131
8,10108,2003-03-03,2003-03-12,2003-03-08,Shipped,NaN,151
9,10109,2003-03-10,2003-03-24,2003-03-11,Shipped,NaN,333


In [9]:
orders_q1 = orders.loc[orders['orderDate'] < '2003-02-01', :].copy()
orders_q2 = orders.loc[orders['orderDate'] >= '2003-02-01', :].copy()

result_keyed = pd.concat([orders_q1, orders_q2], keys = ['Q1', 'Q2']) # 기본 값이 axis=0 이다
result_keyed

orderNumber  orderDate requiredDate shippedDate   status  comments  customerNumber
Q1 0        10100 2003-01-06   2003-01-13  2003-01-10  Shipped       NaN             363
   1        10101 2003-01-09   2003-01-18  2003-01-11  Shipped       NaN             128
   2        10102 2003-01-10   2003-01-18  2003-01-14  Shipped       NaN             181
   3        10103 2003-01-29   2003-02-07  2003-02-02  Shipped       NaN             121
   4        10104 2003-01-31   2003-02-14  2003-01-29  Shipped       NaN             141
Q2 5        10105 2003-02-11   2003-02-21  2003-02-12  Shipped       NaN             145
   6        10106 2003-02-17   2003-02-24  2003-02-21  Shipped       NaN             278
   7        10107 2003-02-24   2003-03-03  2003-02-26  Shipped       NaN             131
   8        10108 2003-03-03   2003-03-12  2003-03-08  Shipped       NaN             151
   9        10109 2003-03-10   2003-03-24  2003-03-11  Shipped       NaN             333

In [12]:
# 코드
# axis= 1 => 열 방향 (가로로 붙이기)
cust_basic = customers.loc[:, ['customerNumber', 'customerName', 'country']].set_index('customerNumber')

pay_total = payments.groupby('customerNumber')['amount'].sum().rename('totalPaid')

result_cols = pd.concat([cust_basic, pay_total], axis=1, join='outer')
result_cols

,customerName,country,totalPaid
customerNumber,,,
103,Atelier graphique,France,20638.22
112,Signal Gift Stores,USA,32641.98
114,"Australian Collectors, Co.",Australia,6066.78
119,La Rochelle Gifts,France,33347.88
121,Baane Mini Imports,Norway,44400.50
124,Mini Gifts Distributors Ltd.,USA,37281.34
128,"Blauer See Auto, Co.",Germany,14191.12
129,Mini Wheels Co.,USA,20009.53
131,Land of Toys Inc.,USA,10223.83


In [13]:
# 코드
# join = 'inner' vs 'outer' 비교
# outer : 양쪽 인덱스의 합집합 => NaN 발생 가능
# inner : 양쪽 인덱스의 교집합 => NaN 발생 없음

outer_c = pd. concat([cust_basic, pay_total], axis=1, join='outer')
inner_c = pd. concat([cust_basic, pay_total], axis=1, join='inner')

outer_c.shape, inner_c.shape

((10, 3), (9, 3))

---
## 2. `pd.merge()` — INNER JOIN (기본값)

> **언제 쓰나?** 양쪽 테이블에 **모두 존재하는** 행만 필요할 때

```python
pd.merge(left, right, how='inner', on=None, left_on=None, right_on=None)
```

```
orders        customers
----------    ----------
orderNumber   customerNumber  <- 공통 키
customerNumber
...           customerName
              country
```


In [14]:
# 코드
inner = pd.merge(
    orders, customers.loc[:, ['customerNumber', 'customerName', 'country', 'creditLimit']],
    on = 'customerNumber', # 조인 키 (컬럼명이 같을 때 on 사용)
    how = 'inner'
)

inner

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber,customerName,country,creditLimit
0,10101,2003-01-09,2003-01-18,2003-01-11,Shipped,NaN,128,"Blauer See Auto, Co.",Germany,59700
1,10103,2003-01-29,2003-02-07,2003-02-02,Shipped,NaN,121,Baane Mini Imports,Norway,81700
2,10104,2003-01-31,2003-02-14,2003-01-29,Shipped,NaN,141,Euro+ Shopping Channel,Spain,227600
3,10107,2003-02-24,2003-03-03,2003-02-26,Shipped,NaN,131,Land of Toys Inc.,USA,114900


---
## 3. `pd.merge()` — LEFT JOIN

> **언제 쓰나?** **왼쪽 테이블의 모든 행**을 유지하면서,
> 오른쪽 정보가 없으면 `NaN` 채우기

```
LEFT  : orders    (모두 보존)
RIGHT : customers (매칭되는 것만)
-> 주문이 있어도 customers 에 없는 customerNumber -> NaN
```


In [17]:
left_df = pd.merge(
    orders, customers.loc[:, ['customerNumber', 'customerName', 'country', 'creditLimit']],
    on = 'customerNumber', # 조인 키 (컬럼명이 같을 때 on 사용)
    how = 'left'
)

left_df.loc[:, ['customerName', 'creditLimit']].isna().sum()

,0
customerName,6
creditLimit,6


---
## 4. `pd.merge()` — RIGHT JOIN

> **언제 쓰나?** **오른쪽 테이블의 모든 행**을 유지하면서,
> 왼쪽 정보가 없으면 `NaN` 채우기

```
LEFT  : orders    (매칭되는 것만)
RIGHT : customers (모두 보존)
-> 주문이 한 건도 없는 고객도 결과에 포함
```


In [18]:
right_df = pd.merge(
    orders, customers.loc[:, ['customerNumber', 'customerName', 'country', 'creditLimit']],
    on = 'customerNumber', # 조인 키 (컬럼명이 같을 때 on 사용)
    how = 'right'
)

right_df

,orderNumber,orderDate,requiredDate,shippedDate,status,comments,customerNumber,customerName,country,creditLimit
0,NaN,NaT,NaN,NaT,NaN,NaN,103,Atelier graphique,France,21000
1,NaN,NaT,NaN,NaT,NaN,NaN,112,Signal Gift Stores,USA,71800
2,NaN,NaT,NaN,NaT,NaN,NaN,114,"Australian Collectors, Co.",Australia,117300
3,NaN,NaT,NaN,NaT,NaN,NaN,119,La Rochelle Gifts,France,118200
4,10103.0,2003-01-29,2003-02-07,2003-02-02,Shipped,NaN,121,Baane Mini Imports,Norway,81700
5,NaN,NaT,NaN,NaT,NaN,NaN,124,Mini Gifts Distributors Ltd.,USA,210500
6,10101.0,2003-01-09,2003-01-18,2003-01-11,Shipped,NaN,128,"Blauer See Auto, Co.",Germany,59700
7,NaN,NaT,NaN,NaT,NaN,NaN,129,Mini Wheels Co.,USA,64600
8,10107.0,2003-02-24,2003-03-03,2003-02-26,Shipped,NaN,131,Land of Toys Inc.,USA,114900
9,10104.0,2003-01-31,2003-02-14,2003-01-29,Shipped,NaN,141,Euro+ Shopping Channel,Spain,227600


---
## 5. `pd.merge()` — OUTER JOIN (FULL OUTER)

> **언제 쓰나?** 양쪽 테이블의 **모든 행을 남기고** 싶을 때
> 매칭이 안 되는 쪽은 `NaN` 으로 채워짐

```
INNER  ⊂  LEFT  ⊂  OUTER
INNER  ⊂  RIGHT ⊂  OUTER
```


In [22]:
# 코드
outer_df = pd.merge(
    orders,
    customers,
    on='customerNumber',
    how='outer'
)

# outer_df
print(f"주문없는 고객: {outer_df['orderNumber'].isna().sum()}건")
print(f"고객정보 없는 주문: {outer_df['customerNumber'].isna().sum()}건")

주문없는 고객: 6건
고객정보 없는 주문: 0건


---
### JOIN 유형 비교 요약

| JOIN 유형 | 결과에 포함되는 행 | NaN 발생 |
|-----------|------------------|---------|
| `inner`   | 양쪽 모두 키가 있는 행만 | 없음 |
| `left`    | 왼쪽 전부 + 오른쪽 매칭 | 오른쪽 |
| `right`   | 오른쪽 전부 + 왼쪽 매칭 | 왼쪽 |
| `outer`   | 양쪽 전부 | 양쪽 |


---
## 6. 다중 키 & 체인 merge

### 6-1. 다중 키 merge
> 두 컬럼을 동시에 키로 사용 — `on=["col1", "col2"]`


In [ ]:
# 코드

,orderNumber,productName,productLine,quantityOrdered,priceEach,margin_pct
0,10100,1937 Lincoln Berline,Vintage Cars,30,136.00,79.4
1,10100,1998 Chrysler Plymouth Prowler,Classic Cars,50,101.26,-0.2
2,10100,1970 Dodge Coronet,Classic Cars,22,65.00,48.7
3,10101,1932 Alfa Romeo 8C2300 Spider S...,Classic Cars,25,75.99,75.7
4,10101,1982 Camaro Z28,Classic Cars,26,45.25,-10.9
5,10102,1969 Harley Davidson Ultimate C...,Motorcycles,39,95.70,96.1
6,10102,1952 Alpine Renault 1300,Classic Cars,41,214.30,117.4
7,10103,18th Century Vintage Horse Carr...,Vintage Cars,20,60.74,0.0
8,10103,1969 Corvette Z28,Classic Cars,18,65.00,38.6
9,10104,1937 Lincoln Berline,Vintage Cars,35,136.00,79.4


### 6-2. left_on / right_on — 컬럼명이 다를 때


In [27]:
employees2 = employees.loc[:, ['employeeNumber', 'firstName', 'lastName', 'jobTitle']].rename(columns={
        'employeeNumber' : 'mgrNumber', 'firstName' : 'mgrFirstName', 'lastName' : 'mgrLastName', 'jobTitle' : 'mgrTitle'
    })

In [28]:
# 셀프조인 문법
# reportsTo => employeeNumber

emp_with_manager = pd.merge(
    employees,
    employees2,
    left_on = 'reportsTo',
    right_on = 'mgrNumber',
    how='left'
)

emp_with_manager

,employeeNumber,lastName,firstName,extension,email,officeCode,reportsTo,jobTitle,mgrNumber,mgrFirstName,mgrLastName,mgrTitle
0,1002,Murphy,Diane,x5800,dmurphy@classicmodelcars.com,1,NaN,President,NaN,NaN,NaN,NaN
1,1056,Patterson,Mary,x4611,mpatterso@classicmodelcars.com,6,1002.0,VP Sales,1002.0,Diane,Murphy,President
2,1076,Firrelli,Julie,x9273,jfirrelli@classicmodelcars.com,2,1002.0,VP Marketing,1002.0,Diane,Murphy,President
3,1088,Patterson,Jeff,x4871,jpatterson@classicmodelcars.com,3,1002.0,Sales Manager (NA),1002.0,Diane,Murphy,President
4,1102,Bondur,Gerard,x5408,gbondur@classicmodelcars.com,4,1056.0,Sales Manager (EMEA),1056.0,Mary,Patterson,VP Sales
5,1143,Bow,Anthony,x5428,abow@classicmodelcars.com,1,1056.0,Sales Manager (APAC),1056.0,Mary,Patterson,VP Sales
6,1165,Jennings,Leslie,x3291,ljennings@classicmodelcars.com,3,1143.0,Sales Rep,1143.0,Anthony,Bow,Sales Manager (APAC)
7,1166,Thompson,Leslie,x4065,lthompson@classicmodelcars.com,4,1143.0,Sales Rep,1143.0,Anthony,Bow,Sales Manager (APAC)
8,1188,Firrelli,Julie,x2173,jfirrelli@classicmodelcars.com,2,1143.0,Sales Rep,1143.0,Anthony,Bow,Sales Manager (APAC)


### 6-3. 체인 merge — 3개 이상 테이블 연결


In [35]:
chain_df = (orderdetails
            .merge(products, on = 'productCode')
            .merge(productlines, on = 'productLine'))

# 카테고리별 매출 합계 (답지 확인)
chain_df.head()



,orderNumber,productCode,quantityOrdered,priceEach,orderLineNumber,productName,productLine,productScale,productVendor,productDescription,quantityInStock,buyPrice,MSRP,textDescription,htmlDescription,image
0,10100,S18_1342,30,136.00,3,1937 Lincoln Berline,Vintage Cars,1:18,RC OPUS,Detailed exterior paint job in ...,8693,75.82,135.99,Our Vintage Car models realisti...,NaN,NaN
1,10100,S18_2238,50,101.26,2,1998 Chrysler Plymouth Prowler,Classic Cars,1:18,Carousel DieCast Legends,Uniquely styled in yellow with ...,9274,101.51,149.99,Attention car enthusiasts: Make...,NaN,NaN
2,10100,S24_1444,22,65.00,4,1970 Dodge Coronet,Classic Cars,1:24,Highway 66 Mini Classics,Detailed color and decal trim.,3420,43.71,65.00,Attention car enthusiasts: Make...,NaN,NaN
3,10101,S18_4409,25,75.99,1,1932 Alfa Romeo 8C2300 Spider S...,Classic Cars,1:18,Exoto Designs,"Features opening hood, doors, t...",9042,43.26,143.00,Attention car enthusiasts: Make...,NaN,NaN
4,10101,S24_2840,26,45.25,2,1982 Camaro Z28,Classic Cars,1:24,Second Gear Diecast,"Features include opening trunk,...",4772,50.81,131.44,Attention car enthusiasts: Make...,NaN,NaN


---
## 7. `DataFrame.join()` — 인덱스 기반 조인

> **언제 쓰나?** 인덱스를 공유하는 테이블끼리 빠르게 붙일 때
> `merge(left_index=True, right_index=True)` 의 편의 문법

```python
left.join(right, on=None, how='left', lsuffix='', rsuffix='')
```


In [42]:
# products 인덱스 => productCode
# productlines 인덱스 => productLine
prod_idx = products.set_index('productCode')[['productName', 'productLine', 'buyPrice', 'MSRP']]

pline_idx = productlines.set_index('productLine')[['textDescription']]

joined = prod_idx.join(pline_idx, on = 'productLine', how = 'left')
joined

,productName,productLine,buyPrice,MSRP,textDescription
productCode,,,,,
S10_1678,1969 Harley Davidson Ultimate C...,Motorcycles,48.81,95.70,Our motorcycles are state of th...
S10_1949,1952 Alpine Renault 1300,Classic Cars,98.58,214.30,Attention car enthusiasts: Make...
S12_1099,18th Century Vintage Horse Carr...,Vintage Cars,60.74,180.00,Our Vintage Car models realisti...
S12_3148,1969 Corvette Z28,Classic Cars,46.91,99.99,Attention car enthusiasts: Make...
S18_1342,1937 Lincoln Berline,Vintage Cars,75.82,135.99,Our Vintage Car models realisti...
S18_2238,1998 Chrysler Plymouth Prowler,Classic Cars,101.51,149.99,Attention car enthusiasts: Make...
S18_4409,1932 Alfa Romeo 8C2300 Spider S...,Classic Cars,43.26,143.00,Attention car enthusiasts: Make...
S24_1444,1970 Dodge Coronet,Classic Cars,43.71,65.00,Attention car enthusiasts: Make...
S24_2840,1982 Camaro Z28,Classic Cars,50.81,131.44,Attention car enthusiasts: Make...


In [37]:
productlines.head(1)

,productLine,textDescription,htmlDescription,image
0,Classic Cars,Attention car enthusiasts: Make...,NaN,NaN


In [ ]:
# 코드

,firstName,lastName,jobTitle,officeCode,city,country,territory
employeeNumber,,,,,,,
1002,Diane,Murphy,President,1,San Francisco,USA,NaN
1056,Mary,Patterson,VP Sales,6,Sydney,Australia,APAC
1076,Julie,Firrelli,VP Marketing,2,Boston,USA,NaN
1088,Jeff,Patterson,Sales Manager (NA),3,NYC,USA,NaN
1102,Gerard,Bondur,Sales Manager (EMEA),4,Paris,France,EMEA
1143,Anthony,Bow,Sales Manager (APAC),1,San Francisco,USA,NaN
1165,Leslie,Jennings,Sales Rep,3,NYC,USA,NaN
1166,Leslie,Thompson,Sales Rep,4,Paris,France,EMEA
1188,Julie,Firrelli,Sales Rep,2,Boston,USA,NaN


---
## 8. `pd.merge_ordered()` — 정렬 보장 병합

> **언제 쓰나?** 시계열/순서가 있는 데이터를 병합하되,
> **결과를 정렬된 순서로** 유지하고 싶을 때
> `fill_method='ffill'` 로 누락값을 앞 값으로 채우기 가능

```python
pd.merge_ordered(left, right, on=None, left_on=None, right_on=None,
                 fill_method=None, how='outer')
```


In [ ]:
# 코드

/tmp/ipykernel_1994/2827233607.py:7: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  ordered = pd.merge_ordered(


,customerNumber,orderDate,orderNumber,paymentDate,amount
0,363,2003-01-06,10100.0,NaT,NaN
1,128,2003-01-09,10101.0,NaT,NaN
2,128,NaT,NaN,2003-02-14,14191.12
3,181,2003-01-10,10102.0,NaT,NaN
4,121,2003-01-29,10103.0,NaT,NaN
5,121,NaT,NaN,2004-11-09,44400.50
6,141,2003-01-31,10104.0,NaT,NaN
7,145,2003-02-11,10105.0,NaT,NaN
8,278,2003-02-17,10106.0,NaT,NaN
9,131,2003-02-24,10107.0,NaT,NaN


---
## 9. `pd.merge_asof()` — 근사값(asof) 병합

> **언제 쓰나?** 정확히 일치하는 키가 없을 때,
> **가장 가까운 이전(또는 이후) 값**으로 매칭
> 예) 주문일 이전에 가장 최근에 한 결제 찾기, 주가 데이터와 이벤트 연결

```python
pd.merge_asof(left, right, on=None, by=None,
              direction='backward')   # 'backward' | 'forward' | 'nearest'
```
> 주의: 두 DataFrame 모두 `on` 컬럼 기준으로 **정렬되어 있어야 함**


In [ ]:
# 코드

각 주문의 직전 결제 정보:


,orderNumber,customerNumber,orderDate,paymentDate,amount
0,10100,363,2003-01-06,NaT,NaN
1,10101,128,2003-01-09,NaT,NaN
2,10102,181,2003-01-10,NaT,NaN
3,10103,121,2003-01-29,NaT,NaN
4,10104,141,2003-01-31,NaT,NaN
5,10105,145,2003-02-11,NaT,NaN
6,10106,278,2003-02-17,NaT,NaN
7,10107,131,2003-02-24,NaT,NaN
8,10108,151,2003-03-03,NaT,NaN
9,10109,333,2003-03-10,NaT,NaN


In [ ]:
# 코드

,orderNumber,orderDate,paymentDate,amount
0,10100,2003-01-06,NaT,NaN
1,10101,2003-01-09,2003-02-14,14191.12
2,10102,2003-01-10,NaT,NaN
3,10103,2003-01-29,2004-11-09,44400.50
4,10104,2003-01-31,NaT,NaN
5,10105,2003-02-11,NaT,NaN
6,10106,2003-02-17,NaT,NaN
7,10107,2003-02-24,2004-07-09,10223.83
8,10108,2003-03-03,NaT,NaN
9,10109,2003-03-10,NaT,NaN


---
## 10. `combine_first()` — 결측값 보완

> **언제 쓰나?** DataFrame A 에 `NaN` 이 있을 때, DataFrame B 의 값으로 채우기
> A를 우선하고, A가 NaN 인 곳만 B에서 가져옴

```python
df_A.combine_first(df_B)
```


In [ ]:
# 코드

=== source_a (NaN 있음) ===


,productName,buyPrice,MSRP
productCode,,,
S10_1678,1969 Harley Davidson Ultimate C...,48.81,NaN
S10_1949,1952 Alpine Renault 1300,98.58,214.30
S12_1099,18th Century Vintage Horse Carr...,NaN,180.00
S12_3148,1969 Corvette Z28,46.91,99.99


=== source_b (NaN 있음, 가격이 조금 다름) ===


,productName,buyPrice,MSRP
productCode,,,
S10_1678,1969 Harley Davidson Ultimate C...,51.2505,95.70
S10_1949,1952 Alpine Renault 1300,103.5090,NaN
S12_1099,18th Century Vintage Horse Carr...,63.7770,180.00
S12_3148,1969 Corvette Z28,49.2555,99.99


=== combine_first 결과 (a 우선, a의 NaN만 b로 채움) ===


,productName,buyPrice,MSRP
productCode,,,
S10_1678,1969 Harley Davidson Ultimate C...,48.810,95.70
S10_1949,1952 Alpine Renault 1300,98.580,214.30
S12_1099,18th Century Vintage Horse Carr...,63.777,180.00
S12_3148,1969 Corvette Z28,46.910,99.99


---
## 11. `compare()` — 두 DataFrame 차이 비교

> **언제 쓰나?** 두 버전 간 **어떤 값이 바뀌었는지** 빠르게 확인
> 변경된 셀만 표시 (나머지는 `NaN`)

```python
df1.compare(df2, result_names=("v1", "v2"), keep_shape=False)
```


In [ ]:
# 코드

변경된 주문: 3건


status            
                  이전          이후
orderNumber                     
10100        Shipped  Processing
10102        Shipped     On Hold
10105        Shipped   Cancelled

In [ ]:
# 코드

status             customerNumber    
                  이전          이후             이전  이후
orderNumber                                        
10100        Shipped  Processing            NaN NaN
10101            NaN         NaN            NaN NaN
10102        Shipped     On Hold            NaN NaN
10103            NaN         NaN            NaN NaN
10104            NaN         NaN            NaN NaN
10105        Shipped   Cancelled            NaN NaN
10106            NaN         NaN            NaN NaN
10107            NaN         NaN            NaN NaN
10108            NaN         NaN            NaN NaN
10109            NaN         NaN            NaN NaN

---
## 최종 요약 — 언제 어떤 함수를?

| 상황 | 함수 | 핵심 파라미터 |
|------|------|-------------|
| 같은 구조 데이터를 세로로 쌓기 | `pd.concat([a,b])` | `axis=0, ignore_index=True` |
| 같은 인덱스 데이터를 가로로 붙이기 | `pd.concat([a,b], axis=1)` | `join='inner'/'outer'` |
| SQL 스타일 조인 (컬럼 키) | `pd.merge(a, b, on=..., how=...)` | `how='inner/left/right/outer'` |
| 컬럼명이 다를 때 조인 | `pd.merge(a, b, left_on=..., right_on=...)` | — |
| 인덱스 기반 조인 | `a.join(b, on=..., how=...)` | `on=` 으로 컬럼->인덱스 매칭 |
| 정렬 보장 + 시계열 병합 | `pd.merge_ordered(a, b, left_by=...)` | `fill_method='ffill'` |
| 가장 가까운 키로 매칭 | `pd.merge_asof(a, b, direction=...)` | `'backward'/'forward'/'nearest'` |
| NaN 보완 (A 우선) | `a.combine_first(b)` | — |
| 두 버전 차이 확인 | `a.compare(b)` | `result_names=, keep_shape=` |

---
> 참조: [pandas 공식 문서 — Merge, join, concatenate and compare](https://pandas.pydata.org/docs/user_guide/merging.html)
